# Nautiq — Gold de tiempos de espera por eslora y tipo de buque

Origen: `vessel_port_calls_analytics`.

Se generan dos tablas agregadas usando únicamente escalas con `actual_wait_hours` observado:

1. `waiting_avg_per_length`: puerto + rango de eslora.
2. `waiting_avg_per_type`: puerto + rango de eslora + tipo de buque.

No se generan combinaciones vacías: cada fila corresponde a una combinación que realmente existe en los datos.


In [0]:
%run ../setup_nautiq_dev


# Nautiq - setup del entorno

Configuración centralizada para el flujo activo del TFM:

- Acceso seguro a Bronze mediante SAS Token.
- Rutas de los topics AIS.
- Estado técnico de Auto Loader.
- Tablas Silver DEV.
- Gold único para analítica JIT y ML.
- Nombre reservado para el notebook ML de clasificación.
- Cambio futuro entre tablas administradas y ADLS externo.

### Notebooks activos

- `silver_ais_positions_dev`
- `silver_ais_static_dev`
- `gold_vessel_jit_features`
- `ml_vessel_jit_classification` *(se implementará después)*


DataFrame[]

NAUTIQ - CONFIGURACION DEL ENTORNO
Environment: dev
Catalog: masterxyz002dbr
Target storage mode: managed
Ops volume: /Volumes/masterxyz002dbr/ops/nautiq_dev

[OK] ais_positions: 16 elementos encontrados
[OK] ais_static: 24 elementos encontrados

Silver DEV tables:
  - masterxyz002dbr.silver.ais_positions_dev
  - masterxyz002dbr.silver.ais_static_dev

Gold table:
  - masterxyz002dbr.gold.vessel_jit_features

Active notebooks:
  - silver_ais_positions_dev
  - silver_ais_static_dev
  - gold_vessel_jit_features
  - ml_vessel_jit_classification  [future]

[OK] Setup completado correctamente.


In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
source_table = vessel_port_calls_target_table

length_table = waiting_avg_per_length_target_table
length_path = waiting_avg_per_length_target_path

type_table = waiting_avg_per_type_target_table
type_path = waiting_avg_per_type_target_path

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")
print("Source:", source_table)
print("Length:", length_table)
print("Type:", type_table)


Source: masterxyz002dbr.gold.vessel_port_calls_analytics
Length: masterxyz002dbr.gold.waiting_avg_per_length
Type: masterxyz002dbr.gold.waiting_avg_per_type


In [0]:
# ============================================================
# 1. ESCALAS REALES CON ESPERA OBSERVADA
# ============================================================

if not spark.catalog.tableExists(source_table): raise RuntimeError(f"No existe la Gold origen: {source_table}")
port_calls = spark.table(source_table).filter(F.col("actual_wait_hours").isNotNull()).filter(F.col("actual_wait_hours") >= 0).filter(F.col("vessel_length_band").isNotNull()).filter(F.col("vessel_length_band") != "OTHER")

display(port_calls.agg(F.count("*").alias("ml_eligible_calls"), F.countDistinct("mmsi").alias("ml_eligible_vessels"), F.round(F.avg("actual_wait_hours"), 2).alias("avg_wait_hours"), F.round(F.expr("percentile_approx(actual_wait_hours, 0.5)"), 2).alias("median_wait_hours")))


ml_eligible_calls,ml_eligible_vessels,avg_wait_hours,median_wait_hours
169,110,16.78,21.93


In [0]:
# ============================================================
# 2. WAITING AVG POR PUERTO + ESLORA
# ============================================================

waiting_avg_per_length = port_calls.groupBy("destination_port_code", "vessel_length_band").agg(F.count("*").alias("port_calls"), F.countDistinct("mmsi").alias("vessels"), F.round(F.avg("actual_wait_hours"), 2).alias("avg_wait_hours"), F.round(F.expr("percentile_approx(actual_wait_hours, 0.50)"), 2).alias("median_wait_hours"), F.round(F.expr("percentile_approx(actual_wait_hours, 0.25)"), 2).alias("p25_wait_hours"), F.round(F.expr("percentile_approx(actual_wait_hours, 0.75)"), 2).alias("p75_wait_hours"), F.round(F.min("actual_wait_hours"), 2).alias("min_wait_hours"), F.round(F.max("actual_wait_hours"), 2).alias("max_wait_hours")).withColumn("gold_processed_timestamp", F.current_timestamp())

write_delta_table(df=waiting_avg_per_length, target_table=length_table, target_path=length_path)
print(f"[OK] Gold creada: {length_table}")


[OK] Gold creada: masterxyz002dbr.gold.waiting_avg_per_length


In [0]:
# ============================================================
# 3. WAITING AVG POR PUERTO + ESLORA + TIPO DE BUQUE
# Solo aparecen categorias que existen realmente en los datos.
# ============================================================

waiting_avg_per_type = port_calls.filter(F.col("ship_type_category").isNotNull()).groupBy("destination_port_code", "vessel_length_band", "ship_type_category").agg(F.count("*").alias("port_calls"), F.countDistinct("mmsi").alias("vessels"), F.round(F.avg("actual_wait_hours"), 2).alias("avg_wait_hours"), F.round(F.expr("percentile_approx(actual_wait_hours, 0.50)"), 2).alias("median_wait_hours"), F.round(F.expr("percentile_approx(actual_wait_hours, 0.25)"), 2).alias("p25_wait_hours"), F.round(F.expr("percentile_approx(actual_wait_hours, 0.75)"), 2).alias("p75_wait_hours"), F.round(F.min("actual_wait_hours"), 2).alias("min_wait_hours"), F.round(F.max("actual_wait_hours"), 2).alias("max_wait_hours")).withColumn("gold_processed_timestamp", F.current_timestamp())

write_delta_table(df=waiting_avg_per_type, target_table=type_table, target_path=type_path)
print(f"[OK] Gold creada: {type_table}")


[OK] Gold creada: masterxyz002dbr.gold.waiting_avg_per_type


In [0]:
# ============================================================
# 4. RESULTADOS
# ============================================================

display(spark.table(length_table).orderBy("destination_port_code", "vessel_length_band"))
display(spark.table(type_table).orderBy("destination_port_code", "vessel_length_band", F.desc("port_calls"), "ship_type_category"))


destination_port_code,vessel_length_band,port_calls,vessels,avg_wait_hours,median_wait_hours,p25_wait_hours,p75_wait_hours,min_wait_hours,max_wait_hours,gold_processed_timestamp
ESALG,100-200,8,8,12.98,8.07,0.0,25.69,0.0,30.0,2026-08-25T11:09:35.901Z
ESALG,200-300,25,24,19.3,29.37,3.18,29.86,0.0,32.61,2026-08-25T11:09:35.901Z
ESALG,300-600,19,17,15.53,16.11,1.71,29.79,0.0,30.01,2026-08-25T11:09:35.901Z
ESBCN,100-200,19,17,12.42,4.9,0.0,29.74,0.0,35.32,2026-08-25T11:09:35.901Z
ESBCN,200-300,23,22,17.1,26.09,0.0,29.89,0.0,32.29,2026-08-25T11:09:35.901Z
ESBCN,300-600,11,10,9.27,1.96,0.0,23.02,0.0,29.91,2026-08-25T11:09:35.901Z
ESVLC,100-200,20,20,24.42,29.48,0.0,29.94,0.0,70.92,2026-08-25T11:09:35.901Z
ESVLC,200-300,25,23,19.77,27.21,0.0,29.85,0.0,75.83,2026-08-25T11:09:35.901Z
ESVLC,300-600,19,19,12.67,2.12,0.0,29.49,0.0,31.15,2026-08-25T11:09:35.901Z


destination_port_code,vessel_length_band,ship_type_category,port_calls,vessels,avg_wait_hours,median_wait_hours,p25_wait_hours,p75_wait_hours,min_wait_hours,max_wait_hours,gold_processed_timestamp
ESALG,100-200,Carga,8,8,12.98,8.07,0.0,25.69,0.0,30.0,2026-08-25T11:09:43.167Z
ESALG,200-300,Carga,25,24,19.3,29.37,3.18,29.86,0.0,32.61,2026-08-25T11:09:43.167Z
ESALG,300-600,Carga,19,17,15.53,16.11,1.71,29.79,0.0,30.01,2026-08-25T11:09:43.167Z
ESBCN,100-200,Carga,19,17,12.42,4.9,0.0,29.74,0.0,35.32,2026-08-25T11:09:43.167Z
ESBCN,200-300,Carga,23,22,17.1,26.09,0.0,29.89,0.0,32.29,2026-08-25T11:09:43.167Z
ESBCN,300-600,Carga,11,10,9.27,1.96,0.0,23.02,0.0,29.91,2026-08-25T11:09:43.167Z
ESVLC,100-200,Carga,20,20,24.42,29.48,0.0,29.94,0.0,70.92,2026-08-25T11:09:43.167Z
ESVLC,200-300,Carga,25,23,19.77,27.21,0.0,29.85,0.0,75.83,2026-08-25T11:09:43.167Z
ESVLC,300-600,Carga,19,19,12.67,2.12,0.0,29.49,0.0,31.15,2026-08-25T11:09:43.167Z


In [0]:
# ============================================================
# 5. CONTROL: LAS TABLAS AGREGAN LOS MISMOS CASOS REALES
# ============================================================

source_calls = port_calls.count()
length_calls = spark.table(length_table).agg(F.sum("port_calls").alias("n")).first()["n"]
type_calls = spark.table(type_table).agg(F.sum("port_calls").alias("n")).first()["n"]

if source_calls != length_calls: raise RuntimeError(f"waiting_avg_per_length no cuadra: {source_calls} vs {length_calls}")
if source_calls != type_calls: raise RuntimeError(f"waiting_avg_per_type no cuadra: {source_calls} vs {type_calls}")
print(f"[OK] Ambas tablas agregan {source_calls} escalas reales.")


[OK] Ambas tablas agregan 169 escalas reales.


## Uso posterior

`waiting_avg_per_length` es el baseline más estable para el primer modelo porque fragmenta menos la muestra.

`waiting_avg_per_type` permite un análisis más detallado por `destination_port_code + vessel_length_band + ship_type_category`; al no crear una matriz artificial de categorías, solo aparecen grupos con al menos una escala observada.

